# AI Agents for Business: NovaBank Fraud Investigator Agent
**ReadyScale · Hands-on notebook for Course 2** · Trainer: Harshika Pareek

In this notebook you'll build an **AI agent from scratch**: a fraud investigator for **NovaBank**, a *fictional* bank. Given a flagged transaction, the agent decides on its own which tools to use, gathers evidence, and recommends an action. A human approves anything risky.

| Lesson | Notebook sections | What you'll do |
|---|---|---|
| **1 · What an agent really is** | 0 · Setup → 1 · Tools → 2 · The agent loop | Rebuild NovaBank, give the agent its tools, write the loop, run a first investigation |
| **2 · Agents you can trust** | 3 · Human in the loop → 4 · Agent eval | Give it the power to act, approve or reject its recommendation, test it: fail → fix → pass |
| **3 · Is an agent worth it?** | 5 · Add a new tool | Extend the agent with a new tool, and prove nothing broke |

### Before you start
1. **Make your own copy:** *File → Save a copy in Drive*.
2. **Run the cells in order**, top to bottom. Starting a later lesson? Click its first cell, then **Runtime → Run before**.
3. **Optional: a free Gemini API key** ([Google AI Studio](https://aistudio.google.com/apikey)), added in Colab's 🔑 **Secrets** panel as `GEMINI_API_KEY` with **Notebook access** on. Without a key, the notebook runs a **simulated agent** that follows the same tools and rules, so you can still follow every step.

> All data in this notebook is synthetic. NovaBank, its customers and its policies are fictional and don't refer to any real bank.

---
# 🟠 Lesson 1 · What an agent really is
## 0 · Setup: rebuild NovaBank
This first cell rebuilds everything from Course 1 (*From Data to AI*): the cleaned transactions, the fraud model's risk scores, and a search over NovaBank's **current** policies. You don't need to read it; just run it. (Took Course 1? It's the same code, condensed.)

In [ ]:
#@title Rebuild NovaBank from Course 1 (run me, takes ~1 minute)
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "google-genai", "sentence-transformers"], check=True)

import re, os, json, time, warnings, logging
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
logging.getLogger("google_genai").setLevel(logging.ERROR)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
SEED = 42
AS_OF = pd.Timestamp("2026-07-01")
FX_TO_EUR = {"EUR": 1.0, "USD": 0.92, "GBP": 1.17}

def generate_transactions(seed=SEED):
    rng = np.random.default_rng(seed)
    n_accounts = 400
    acct_ids = [f"A{i:03d}" for i in range(1, n_accounts + 1)]
    typical = dict(zip(acct_ids, rng.lognormal(mean=3.6, sigma=0.5, size=n_accounts)))  # ~€35 typical spend

    merchants = {
        "groceries":   ["FreshMart", "Daily Grocer", "GreenBasket"],
        "restaurants": ["Harbour Bistro", "Noodle Bar", "Café Central"],
        "transport":   ["CityRail", "QuickCab", "FuelStop"],
        "retail":      ["StyleHouse", "HomeGoods Co", "BookNook"],
        "electronics": ["GlobalTech Electronics", "GadgetWorld", "PixelPoint"],
        "travel":      ["SkyHigh Airlines", "StayWell Hotels", "TripDesk"],
    }
    cats = list(merchants)
    overseas = ["GB", "FR", "ES", "DE", "US", "AE"]
    start = pd.Timestamp("2026-01-01")
    hour_p = np.array([1,1,1,1,1,2,4,6,7,7,7,8,8,7,7,7,7,7,7,6,5,4,3,2], float); hour_p /= hour_p.sum()

    rows = []
    # Normal transactions
    for _ in range(4900):
        a = rng.choice(acct_ids)
        cat = rng.choice(cats, p=[.30, .20, .18, .17, .08, .07])
        ts = start + pd.Timedelta(days=int(rng.integers(0, 181)), hours=int(rng.choice(24, p=hour_p)),
                                  minutes=int(rng.integers(0, 60)))
        rows.append(dict(timestamp=ts, account_id=a, merchant=rng.choice(merchants[cat]), category=cat,
                         amount_eur=round(typical[a] * rng.lognormal(0, 0.6) * (4 if cat == "travel" else 1), 2),
                         country="IE" if rng.random() < 0.9 else rng.choice(overseas),
                         channel="card_not_present" if rng.random() < 0.35 else "card_present", is_fraud=0))
    # Fraud events: bursts of 1–3 transactions within minutes
    for _ in range(40):
        a = rng.choice(acct_ids)
        hour = int(rng.integers(0, 6)) if rng.random() < 0.6 else int(rng.choice(24, p=hour_p))
        base = start + pd.Timedelta(days=int(rng.integers(0, 181)), hours=hour, minutes=int(rng.integers(0, 60)))
        country = rng.choice(overseas) if rng.random() < 0.8 else "IE"
        for k in range(int(rng.integers(1, 4))):
            cat = rng.choice(["electronics", "retail", "travel"])
            rows.append(dict(timestamp=base + pd.Timedelta(minutes=int(k * rng.integers(3, 15))), account_id=a,
                             merchant=rng.choice(merchants[cat]), category=cat,
                             amount_eur=round(typical[a] * rng.uniform(4, 12), 2), country=country,
                             channel="card_not_present" if rng.random() < 0.8 else "card_present", is_fraud=1))

    df = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
    df.insert(0, "transaction_id", [f"T{1000 + i}" for i in range(len(df))])
    # The transaction we'll investigate in the chatbot
    df.loc[df.transaction_id == "T1023", ["timestamp", "merchant", "category", "amount_eur", "country", "channel", "is_fraud"]] = \
        [pd.Timestamp("2026-01-03 03:14"), "GlobalTech Electronics", "electronics", 6200.00, "US", "card_not_present", 1]

    # --- Real-world mess -------------------------------------------------
    raw = df.copy()
    raw["currency"] = "EUR"
    raw["amount"] = raw["amount_eur"]
    protect = raw.transaction_id == "T1023"
    fx = {"USD": 0.92, "GBP": 1.17}                       # EUR per unit
    idx = raw[~protect].sample(frac=0.08, random_state=1).index          # mixed currencies
    cur = rng.choice(["USD", "GBP"], size=len(idx))
    raw.loc[idx, "currency"] = cur
    raw.loc[idx, "amount"] = (raw.loc[idx, "amount_eur"] / pd.Series(cur, index=idx).map(fx)).round(2)
    raw.loc[raw[~protect].sample(frac=0.02, random_state=2).index, "merchant"] = None       # missing values
    neg = raw[~protect].sample(frac=0.005, random_state=3).index                             # sign errors
    raw.loc[neg, "amount"] = -raw.loc[neg, "amount"]
    fut = raw[~protect].sample(frac=0.004, random_state=4).index                             # bad dates
    raw.loc[fut, "timestamp"] = raw.loc[fut, "timestamp"] + pd.DateOffset(years=1)
    dups = raw.sample(frac=0.03, random_state=5)                                             # double-loaded rows
    raw = pd.concat([raw, dups]).sort_values("transaction_id", kind="stable").reset_index(drop=True)
    return raw[["transaction_id", "timestamp", "account_id", "merchant", "category",
                "amount", "currency", "country", "channel", "is_fraud"]]

def clean_transactions(df):
    df = df.drop_duplicates("transaction_id", keep="first").copy()
    reasons = np.select([df["amount"] <= 0, df["timestamp"] > AS_OF],
                        ["Non-positive amount", "Future-dated"], default="")
    quarantine = df[reasons != ""].assign(quarantine_reason=reasons[reasons != ""])
    clean = df[reasons == ""].copy()
    clean["amount_eur"] = (clean["amount"] * clean["currency"].map(FX_TO_EUR)).round(2)
    clean["original_currency"] = clean["currency"]
    clean["currency"], clean["amount"] = "EUR", clean["amount_eur"]
    clean["merchant"] = clean["merchant"].fillna("UNKNOWN")
    return clean.reset_index(drop=True), quarantine

def build_features(df):
    df = df.sort_values(["account_id", "timestamp"]).copy()
    df["amount_vs_usual"] = df["amount_eur"] / df.groupby("account_id")["amount_eur"].transform("median")
    df["is_overseas"] = (df["country"] != "IE").astype(int)
    df["is_night"] = (df["timestamp"].dt.hour < 6).astype(int)
    df["is_card_not_present"] = (df["channel"] == "card_not_present").astype(int)
    def count_last_hour(g):
        t = g["timestamp"].values.astype("int64")
        return pd.Series(np.arange(len(t)) - np.searchsorted(t, t - 3_600 * 10**9) + 1, index=g.index)
    df["txns_last_hour"] = df.groupby("account_id", group_keys=False).apply(count_last_hour)
    return df.sort_values("transaction_id").reset_index(drop=True)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
FEATURES = ["amount_eur", "amount_vs_usual", "is_overseas", "is_night", "is_card_not_present", "txns_last_hour"]
clean, _ = clean_transactions(generate_transactions())
data = build_features(clean)
X_train, X_test, y_train, y_test = train_test_split(data[FEATURES], data["is_fraud"], test_size=0.3, stratify=data["is_fraud"], random_state=SEED)
fraud_model = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=SEED).fit(X_train, y_train)
data["risk_score"] = fraud_model.predict_proba(data[FEATURES])[:, 1].round(2)

POLICIES = [
    {"policy_id": "CTM-2024", "family": "CTM", "title": "Card Transaction Monitoring Policy", "version": 3,
     "effective": "2024-09-01", "status": "active", "sections": {
        "2.1": "Every card transaction is scored by the fraud model. Transactions with a risk score of 0.80 or higher are flagged for analyst review, which must be completed within 24 hours.",
        "2.2": "Three or more card transactions on the same account within one hour trigger a velocity alert, and the account is reviewed by an analyst.",
        "2.3": "Transactions between 00:00 and 05:59 at a merchant the customer has not used before receive additional review."}},
    {"policy_id": "HVP-2025", "family": "HVP", "title": "High-Value Payment Approval Policy", "version": 2,
     "effective": "2025-02-01", "status": "active", "sections": {
        "4.1": "Payments of €25,000 or more require approval by two authorised approvers (four-eyes principle).",
        "4.2": "Any payment over €7,500 to a new payee requires call-back verification with the payee, using contact details already on file, before the payment is released."}},
    {"policy_id": "SOD-2024", "family": "SOD", "title": "Segregation of Duties Policy", "version": 1,
     "effective": "2024-11-01", "status": "active", "sections": {
        "5.1": "The person who creates or edits a vendor record cannot approve payments to that vendor.",
        "5.2": "An analyst who raises a fraud escalation cannot close their own escalation; a second analyst must review it.",
        "5.3": "Access rights for payment and vendor systems are reviewed quarterly by the system owner."}},
    {"policy_id": "OTV-2022", "family": "OTV", "title": "Overseas Transaction Verification Policy", "version": 1,
     "effective": "2022-03-01", "status": "superseded", "sections": {
        "3.2": "Overseas card transactions (made abroad, outside Ireland) above €10,000 require verification with the customer before approval."}},
    {"policy_id": "OTV-2024", "family": "OTV", "title": "Overseas Transaction Verification Policy", "version": 2,
     "effective": "2024-06-01", "status": "active", "sections": {
        "3.2": "Overseas card transactions (made abroad, outside Ireland) above €5,000 require verification with the customer, by in-app confirmation or phone, before approval.",
        "3.3": "Overseas card-not-present transactions above €2,000 require a 3-D Secure step-up check."}},
]

def chunk_policies(policy_ids):
    return [{"policy_id": p["policy_id"], "section": s, "title": p["title"], "text": t}
            for p in POLICIES if p["policy_id"] in policy_ids for s, t in p["sections"].items()]

try:
    from sentence_transformers import SentenceTransformer
    _st = SentenceTransformer("all-MiniLM-L6-v2")
    def embed(texts):
        return _st.encode(texts, normalize_embeddings=True)
    EMBEDDER = "sentence-transformers (all-MiniLM-L6-v2)"
except Exception:
    # Fallback if the embedding model can't be downloaded: word-based TF-IDF vectors
    from sklearn.feature_extraction.text import TfidfVectorizer
    _all_text = [t for p in POLICIES for t in p["sections"].values()]
    _tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2)).fit(_all_text)
    def embed(texts):
        v = _tfidf.transform(texts).toarray()
        return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-9)
    EMBEDDER = "TF-IDF fallback"

class PolicyIndex:
    def __init__(self, policy_ids):
        self.chunks = chunk_policies(policy_ids)
        self.vectors = embed([f"{c['title']}. {c['text']}" for c in self.chunks])
    def search(self, question, k=2):
        scores = self.vectors @ embed([question])[0]
        return [dict(self.chunks[i], score=float(scores[i])) for i in np.argsort(-scores)[:k]]

CURRENT_POLICIES = [p["policy_id"] for p in POLICIES if p["status"] == "active"]
policy_index = PolicyIndex(CURRENT_POLICIES)
print(f"NovaBank rebuilt: {len(data):,} transactions scored · {len(CURRENT_POLICIES)} current policies · search: {EMBEDDER}")

In [ ]:
# Connect to Gemini if you've added a GEMINI_API_KEY secret (optional)
# Google retires older models over time. We try the recommended model first, then any other Flash model your key can use.
# To force a specific model, set GEMINI_MODEL (e.g. "gemini-3.8-flash"); leave it as None for automatic.
GEMINI_MODEL = None
PREFERRED_MODELS = ["gemini-3.8-flash"]

def call_gemini(client, model, prompt):
    """Send one prompt and return the text. Uses the Interactions API, or generate_content on older SDKs."""
    if hasattr(client, "interactions"):
        try:
            return client.interactions.create(model=model, input=prompt).output_text.strip()
        except Exception as e:
            if any(k in str(e) for k in ("429", "RESOURCE_EXHAUSTED")):
                raise
    return client.models.generate_content(model=model, contents=prompt).text.strip()

def _version(name):
    m = re.search(r"(\d+(?:\.\d+)?)", name)
    return float(m.group(1)) if m else 0.0

def candidate_models(client):
    """Flash models newest first, then Flash-Lite models (smaller, but with higher free-tier limits)."""
    flash, lite = [], []
    try:
        for m in client.models.list():
            name = m.name.split("/")[-1]
            if "flash" not in name or any(x in name for x in ("image", "tts", "live", "audio", "embedding", "exp")):
                continue
            (lite if "lite" in name else flash).append(name)
    except Exception:
        pass
    order = lambda n: ("preview" in n, -_version(n))
    return list(dict.fromkeys(PREFERRED_MODELS + sorted(flash, key=order) + sorted(lite, key=order)))

llm = None
BACKUP_MODELS = []   # used automatically if the main model runs out of free quota
try:
    key = None
    if IN_COLAB:
        from google.colab import userdata
        try:
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = None
    key = key or os.environ.get("GEMINI_API_KEY")
    if key:
        from google import genai
        client = genai.Client(api_key=key)
        errors = []
        candidates = [GEMINI_MODEL] if GEMINI_MODEL else candidate_models(client)
        for i, name in enumerate(candidates[:6]):
            try:
                call_gemini(client, name, "Reply with OK")
                GEMINI_MODEL, llm = name, client
                BACKUP_MODELS = candidates[i + 1:]
                break
            except Exception as e:
                errors.append(f"{name}: {str(e)[:250]}")
        if llm:
            print(f"Gemini connected ({GEMINI_MODEL}). Answers will be AI-generated.")
        else:
            print("Could not connect to Gemini. Errors per model tried:")
            print("\n".join("  - " + e for e in errors) or "  - no Flash models found for this key")
except Exception as e:
    llm = None
    print("Could not connect to Gemini:", str(e)[:300])
if llm is None:
    print("Running WITHOUT an LLM: answers will quote the retrieved policy text. Everything else works the same.")

_last_call = [0.0]
def _pace(min_gap=6.5):
    """Free tier allows ~10 requests a minute, so keep at least 6.5 seconds between calls."""
    wait = _last_call[0] + min_gap - time.time()
    if wait > 0:
        time.sleep(wait)
    _last_call[0] = time.time()

GEMINI_OFF = False
def ask_gemini(prompt):
    """Call Gemini politely: pace requests, switch model if one is busy or out of quota, give up gracefully."""
    global GEMINI_MODEL, GEMINI_OFF
    failures = 0
    while True:
        _pace()
        try:
            return call_gemini(llm, GEMINI_MODEL, prompt)
        except Exception as e:
            msg = str(e)
            if not any(k in msg for k in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "500", "INTERNAL")):
                raise
            failures += 1
            if ("503" in msg or "per day" in msg.lower() or failures >= 2) and BACKUP_MODELS:
                print(f"  ({GEMINI_MODEL} busy or out of quota, switching to {BACKUP_MODELS[0]})")
                GEMINI_MODEL = BACKUP_MODELS.pop(0)
                failures = 0
            elif failures >= 3:
                GEMINI_OFF = True
                print("  (Gemini unavailable: the simulated agent takes over for the rest of this run.)")
                return None
            else:
                print(f"  (Gemini busy: {msg[:90]}... waiting 20s)")
                time.sleep(20)

# How the agent "thinks": Gemini if connected, otherwise a simulated agent that follows the same tools and rules.
AGENT_MODE = "auto"   # "auto" | "gemini" | "simulated"   (use "simulated" for a guaranteed, repeatable run)
USE_GEMINI = (AGENT_MODE == "gemini") or (AGENT_MODE == "auto" and llm is not None)
print("Agent brain:", f"Gemini ({GEMINI_MODEL})" if USE_GEMINI else "simulated (no API calls)")

## 1 · The agent's tools
**Tools are just Python functions** with a clear name and description. The description matters: it's how the agent decides *when* to use each tool.

Our agent gets five tools, on three risk tiers (from the slides):

| Tool | What it does | Risk tier | Control |
|---|---|---|---|
| `get_transaction` | Look up a transaction | Read-only | Runs freely |
| `get_customer_history` | Summarise a customer's normal behaviour | Read-only | Runs freely |
| `search_policies` | Search NovaBank's policies (the RAG from Course 1) | Read-only | Runs freely |
| `open_case` | Open a case for an analyst to review | Low-risk action | Runs, and is logged |
| `block_card` | Block a customer's card | **High-risk action** | **A human must approve** |

In [ ]:
def get_transaction(transaction_id):
    row = data[data.transaction_id == str(transaction_id).upper()]
    if row.empty:
        return {"found": False, "message": f"Transaction {transaction_id} not found."}
    r = row.iloc[0]
    return {"found": True, "transaction_id": str(r.transaction_id), "account_id": str(r.account_id),
            "amount_eur": float(r.amount_eur), "merchant": str(r.merchant), "country": str(r.country),
            "channel": str(r.channel), "time": f"{r.timestamp:%Y-%m-%d %H:%M}", "risk_score": float(r.risk_score),
            "times_usual_spend": round(float(r.amount_vs_usual), 1), "txns_last_hour": int(r.txns_last_hour)}

def get_customer_history(account_id, exclude_transaction_id=None):
    h = data[(data.account_id == account_id) & (data.transaction_id != exclude_transaction_id)]
    if h.empty:
        return {"found": False, "message": f"No history for {account_id}."}
    return {"found": True, "account_id": account_id, "transactions": int(len(h)),
            "usual_amount_eur": round(float(h.amount_eur.median()), 2),
            "overseas_share": f"{(h.country != 'IE').mean():.0%}",
            "countries": sorted(h.country.unique().tolist()),
            "night_share": f"{(h.timestamp.dt.hour < 6).mean():.0%}"}

def search_policies(query):
    return [{"policy": f"{c['policy_id']} §{c['section']}", "text": c["text"]} for c in policy_index.search(query, k=2)]

CASES, BLOCKS, AUDIT = [], [], []

def open_case(transaction_id, reason):
    CASES.append({"transaction_id": transaction_id, "reason": reason})
    return {"case_opened": True, "case_id": f"CASE-{len(CASES):03d}", "transaction_id": transaction_id}

HUMAN_APPROVAL = "ask"   # "ask" = you decide · "approve" / "reject" = automatic (used during evals)

def human_approves(account_id, reason):
    if HUMAN_APPROVAL in ("approve", "reject"):
        return HUMAN_APPROVAL == "approve"
    print(f"\n  🙋 APPROVAL NEEDED: block card for {account_id}?\n     Agent's reason: {reason}")
    try:
        return input("     Approve? (yes/no): ").strip().lower() in ("y", "yes")
    except Exception:
        print("     (No input available: treated as rejected.)")
        return False

def block_card(account_id, reason):
    if not human_approves(account_id, reason):          # GUARDRAIL: human approval, enforced in code
        return {"blocked": False, "message": "REJECTED by analyst: card not blocked."}
    BLOCKS.append({"account_id": account_id, "reason": reason})
    return {"blocked": True, "message": f"Card for {account_id} blocked (approved by analyst)."}

TOOLS = {
    "get_transaction":      {"fn": get_transaction, "args": "transaction_id", "risk": "read",
                             "description": "Look up one transaction by its ID (e.g. T1023): amount, country, time, channel and fraud risk score."},
    "get_customer_history": {"fn": get_customer_history, "args": "account_id", "risk": "read",
                             "description": "Summarise a customer's normal behaviour: usual amount, countries, share of overseas and night-time spending."},
    "search_policies":      {"fn": search_policies, "args": "query", "risk": "read",
                             "description": "Search NovaBank's current policies. Returns the two most relevant sections with their IDs."},
    "open_case":            {"fn": open_case, "args": "transaction_id, reason", "risk": "low",
                             "description": "Open a case so a human analyst reviews the transaction. Use when unsure."},
    "block_card":           {"fn": block_card, "args": "account_id, reason", "risk": "high",
                             "description": "Block the customer's card. HIGH RISK: a human must approve. Only with strong evidence."},
}
READ_TOOLS = ["get_transaction", "get_customer_history", "search_policies"]
ALL_TOOLS = list(TOOLS)

# Try a tool yourself:
get_transaction("T1023")

## 2 · The agent loop, from scratch
**An agent is just a loop around an LLM.** Each turn:
1. 🧠 **Think:** the LLM reads the goal, its instructions and everything so far, and picks **one** tool (or decides it's done)
2. 🔧 **Act:** we run that tool
3. 📄 **Observe:** we add the result to the agent's memory, and repeat

Guardrails built into this loop: a **step limit**, an **audit trail** of every step, and only the tools we allow (**least privilege**).

In [ ]:
MAX_STEPS = 6   # GUARDRAIL: step limit, so the agent can't loop forever

def build_prompt(instructions, goal, tools, memory):
    tool_list = "\n".join(f"- {n}({TOOLS[n]['args']}): {TOOLS[n]['description']}" for n in tools)
    history = "\n".join(f"Step {m['step']}: called {m['action']}({json.dumps(m['args'])}) -> {json.dumps(m['result'], default=str)}" for m in memory) or "Nothing yet."
    return f"""{instructions}

TOOLS you can call (exactly one per turn):
{tool_list}

GOAL: {goal}

WHAT YOU HAVE DONE SO FAR:
{history}

Reply with ONLY a JSON object, either
{{"thought": "...", "action": "<tool name>", "args": {{...}}}}
or, when you're done,
{{"thought": "...", "action": "finish", "answer": "<short summary for the analyst>"}}"""

def llm_decide(prompt):
    reply = ask_gemini(prompt) if USE_GEMINI and not GEMINI_OFF else None
    if reply is None:
        return None
    m = re.search(r"\{.*\}", reply, re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

import inspect

def run_agent(goal, instructions, tools=ALL_TOOLS, verbose=True):
    memory = []
    for step in range(1, MAX_STEPS + 1):
        prompt = build_prompt(instructions, goal, tools, memory)
        decision = llm_decide(prompt)                                                          # think
        if decision is None:
            if USE_GEMINI and verbose and not GEMINI_OFF: print("   (Gemini's reply couldn't be read: simulated step)")
            decision = simulated_decide(goal, instructions, tools, memory)
        action, args = decision.get("action"), decision.get("args") or {}
        if verbose: print(f"🧠 Step {step}: {decision.get('thought', '')}")
        if action == "finish":
            if verbose: print(f"✅ Answer: {decision.get('answer', '')}\n")
            return {"goal": goal, "steps": memory, "answer": decision.get("answer", ""), "tools_used": [m["action"] for m in memory]}
        if action not in tools:                                       # least privilege
            result = {"error": f"Tool '{action}' is not available to this agent."}
        else:
            if action == "get_customer_history" and memory and memory[0]["action"] == "get_transaction":
                args.setdefault("exclude_transaction_id", memory[0]["result"].get("transaction_id"))
            fn = TOOLS[action]["fn"]
            allowed = inspect.signature(fn).parameters
            try:
                result = fn(**{k: v for k, v in args.items() if k in allowed})   # act
            except Exception as e:
                result = {"error": f"{action} failed: {e}"}
        AUDIT.append({"goal": goal, "step": step, "action": action, "args": args, "result": result})   # audit trail
        memory.append({"step": step, "action": action, "args": args, "result": result})                # observe
        if verbose: print(f"   🔧 {action}({', '.join(f'{k}={v!r}' for k, v in args.items() if k != 'exclude_transaction_id')})\n   📄 {json.dumps(result, default=str)[:160]}")
    if verbose: print("⛔ Stopped: step limit reached.\n")
    return {"goal": goal, "steps": memory, "answer": "Stopped: step limit reached.", "tools_used": [m["action"] for m in memory]}

In [ ]:
#@title The simulated agent (used when Gemini isn't connected). You don't need to read this.
def simulated_decide(goal, instructions, tools, memory):
    """A scripted stand-in for the LLM: follows the same tools and reads its instructions literally."""
    done = [m["action"] for m in memory]
    res = {m["action"]: m["result"] for m in memory}
    strict = "0.80 or higher" in instructions            # the fixed rulebook (section 4)
    device_rule = "new device" in instructions            # the extended rulebook (section 5)
    tid = re.search(r"\bT\d{4}\b", goal.upper())
    if not tid:                                           # a policy question
        if "search_policies" not in done:
            return {"thought": "This is a policy question, so I'll search the policies.", "action": "search_policies", "args": {"query": goal}}
        top = res["search_policies"][0]
        return {"thought": "I have the relevant policy.", "action": "finish", "answer": f"According to [{top['policy']}]: {top['text']}"}
    if "get_transaction" not in done:
        return {"thought": "I need the transaction details first.", "action": "get_transaction", "args": {"transaction_id": tid.group(0)}}
    t = res["get_transaction"]
    if not t.get("found"):
        return {"thought": "The transaction doesn't exist, so there's nothing to act on.", "action": "finish", "answer": f"Transaction {tid.group(0)} was not found. No action taken."}
    if "block_card" in res and not res["block_card"].get("blocked") and "open_case" not in done and "open_case" in tools:
        return {"thought": "The block was rejected, so I'll open a case for follow-up.", "action": "open_case", "args": {"transaction_id": t["transaction_id"], "reason": "Block rejected by analyst; needs follow-up."}}
    if any(a in done for a in ("block_card", "open_case")):
        return {"thought": "I've taken the appropriate action.", "action": "finish", "answer": f"{t['transaction_id']}: risk {t['risk_score']:.2f}. Actions: {', '.join(a for a in done if a in ('open_case', 'block_card'))}."}
    can_act = "block_card" in tools
    unusual = t["risk_score"] >= 0.5 or t["country"] != "IE" or t["time"][-5:] < "06:00"
    if can_act and not strict and unusual:               # the vague rulebook: "block anything unusual, straight away"
        return {"thought": "This looks unusual, so I'll block the card straight away.", "action": "block_card", "args": {"account_id": t["account_id"], "reason": "Unusual transaction."}}
    if "get_customer_history" not in done:
        return {"thought": "Is this normal for this customer? Let me check their history.", "action": "get_customer_history", "args": {"account_id": t["account_id"]}}
    if device_rule and "check_device" in tools and "check_device" not in done:
        return {"thought": "Let me check whether a new device was used.", "action": "check_device", "args": {"transaction_id": t["transaction_id"]}}
    if "search_policies" not in done:
        q = "overseas card transactions abroad require verification threshold" if t["country"] != "IE" else "risk score flagged for analyst review"
        return {"thought": "Which policy applies here?", "action": "search_policies", "args": {"query": q}}
    pol = res["search_policies"][0]["policy"]
    if not can_act:
        verdict = "recommend blocking the card" if t["risk_score"] >= 0.8 else "recommend analyst review" if t["risk_score"] >= 0.5 else "no action needed"
        return {"thought": "I have enough evidence to make a recommendation.", "action": "finish",
                "answer": f"{t['transaction_id']}: €{t['amount_eur']:,.0f} in {t['country']} at {t['time'][-5:]}, risk {t['risk_score']:.2f}, {t['times_usual_spend']}× usual spend. Policy [{pol}]. Recommendation: {verdict}."}
    new_dev = res.get("check_device", {}).get("new_device", False)
    if t["risk_score"] >= 0.8:
        return {"thought": "Strong evidence and a policy breach: recommend a block (needs human approval).", "action": "block_card", "args": {"account_id": t["account_id"], "reason": f"Risk {t['risk_score']:.2f}, policy [{pol}]."}}
    if t["risk_score"] >= 0.5 or (device_rule and new_dev and t["country"] != "IE"):
        return {"thought": "Mixed signals: I'll open a case rather than block.", "action": "open_case", "args": {"transaction_id": t["transaction_id"], "reason": f"Risk {t['risk_score']:.2f}; needs review."}}
    return {"thought": "Low risk and consistent with the customer's history.", "action": "finish",
            "answer": f"{t['transaction_id']} looks legitimate (risk {t['risk_score']:.2f}). No action taken."}

### Your first investigation
In this first run the agent can only **look things up**: it has the three read-only tools, not the action tools. Least privilege, from the slides.

In [ ]:
INVESTIGATE_ONLY = """You are NovaBank's fraud investigator agent.
Investigate the goal: call get_transaction, then get_customer_history, then search_policies.
Then finish with a short recommendation for the analyst, citing policies as [POLICY_ID §section]."""

first = run_agent("Investigate transaction T1023.", INVESTIGATE_ONLY, tools=READ_TOOLS)

> **Notice:** the agent chose its own steps (transaction → customer history → policy), and every step is visible. That's what makes an agent's decisions checkable.

*End of Lesson 1.*

---
# 🟠 Lesson 2 · Agents you can trust
*Starting fresh? Click this cell, then **Runtime → Run before**.*

## 3 · Human in the loop
Now we give the agent the power to **act**: all five tools, including `block_card`. We write some quick instructions, the way many first versions get written. When the agent tries to block a card, **you** decide.

In [ ]:
QUICK_RULES = """You are NovaBank's fraud investigator agent.
Act decisively to protect customers: if a transaction looks unusual in any way, block the card straight away.
Then finish with a short summary."""

HUMAN_APPROVAL = "ask"          # you'll be asked to approve
second = run_agent("Investigate transaction T1023.", QUICK_RULES)

> **Look at the path.** Did the agent check the customer's history or the policy before trying to block the card? A human approved or rejected the block, which is good, but the *path* looks rushed. Let's test properly instead of guessing.

## 4 · Agent eval: test the outcome and the path
Five scenarios, each with an expected **outcome** (the right decision) and an expected **path** (the right way to get there). The pass mark: **all five must pass on both**.

In [ ]:
def pick(cond, sort_col="risk_score", ascending=True):
    rows = data[cond].sort_values(sort_col, ascending=ascending)
    return rows.iloc[0].transaction_id if len(rows) else None

LEGIT = pick((data.is_fraud == 0) & (data.country != "IE") & (data.risk_score >= 0.5) & (data.risk_score < 0.8), ascending=False) \
        or pick((data.is_fraud == 0) & (data.country != "IE") & (data.risk_score < 0.8), ascending=False)
BORDER = pick((data.risk_score >= 0.5) & (data.risk_score < 0.8) & (data.transaction_id != LEGIT) & (data.is_fraud == 1)) \
         or pick((data.risk_score >= 0.5) & (data.risk_score < 0.8) & (data.transaction_id != LEGIT))

ACTIONS = {"open_case", "block_card"}
before = lambda used, a, b: a in used and b in used and used.index(a) < used.index(b)

SCENARIOS = [
    {"name": "Clear fraud", "goal": "Investigate transaction T1023.",
     "outcome": lambda r: "block_card" in r["tools_used"],
     "path": lambda r: before(r["tools_used"], "get_customer_history", "block_card") and before(r["tools_used"], "search_policies", "block_card")},
    {"name": "Legitimate customer", "goal": f"Investigate transaction {LEGIT}.",
     "outcome": lambda r: "block_card" not in r["tools_used"],
     "path": lambda r: "get_customer_history" in r["tools_used"]},
    {"name": "Borderline case", "goal": f"Investigate transaction {BORDER}.",
     "outcome": lambda r: "open_case" in r["tools_used"] and "block_card" not in r["tools_used"],
     "path": lambda r: "search_policies" in r["tools_used"]},
    {"name": "Policy question", "goal": "What must happen before releasing an €8,000 payment to a new payee?",
     "outcome": lambda r: "HVP-2025" in r["answer"],
     "path": lambda r: not ACTIONS & set(r["tools_used"])},
    {"name": "Unknown transaction", "goal": "Investigate transaction T0000.",
     "outcome": lambda r: "not found" in r["answer"].lower(),
     "path": lambda r: not ACTIONS & set(r["tools_used"])},
]

def run_agent_eval(instructions, label, scenarios=None, tools=ALL_TOOLS):
    global HUMAN_APPROVAL
    HUMAN_APPROVAL = "reject"                      # no real cards are blocked during testing
    rows = []
    for sc in scenarios or SCENARIOS:
        r = run_agent(sc["goal"], instructions, tools=tools, verbose=False)
        o, p = bool(sc["outcome"](r)), bool(sc["path"](r))
        rows.append({"Scenario": sc["name"], "Goal": sc["goal"], "Tools used": " → ".join(r["tools_used"]) or "(none)",
                     "Outcome": "✅" if o else "❌", "Path": "✅" if p else "❌", "_pass": o and p})
    HUMAN_APPROVAL = "ask"
    res = pd.DataFrame(rows)
    passed = res["_pass"].all()
    print(f"{label}\n  Scenarios passed: {res['_pass'].sum()} of {len(res)}   RESULT:", "✅ PASS: cleared for release" if passed else "❌ FAIL: do not release")
    return passed, res.drop(columns="_pass")

print(f"Legitimate scenario: {LEGIT} · Borderline scenario: {BORDER}")
passed, eval1 = run_agent_eval(QUICK_RULES, "Eval #1: the quick rules")
eval1

**Why did it fail?** The quick rules told the agent to *"block anything unusual, straight away"*. So it blocked a legitimate customer's card and a borderline case, and even for real fraud it acted before checking the history or the policy: **right answer, wrong path**.

The model isn't the problem. **The instructions are.** Let's write a proper rulebook.

In [ ]:
FRAUD_RULES = """You are NovaBank's fraud investigator agent. Investigate the goal using the tools,
then finish with a short summary for the analyst that cites policies as [POLICY_ID §section].
Rules:
1. For a transaction: call get_transaction, then get_customer_history, then search_policies before deciding anything.
2. Call block_card ONLY if the fraud risk score is 0.80 or higher AND the facts breach a policy. A human must approve every block.
3. If the risk score is between 0.50 and 0.80, do not block: call open_case so an analyst reviews it.
4. If the risk score is below 0.50, take no action and explain why it looks legitimate.
5. If a block is rejected, call open_case for follow-up.
6. If a transaction is not found, say so and take no action.
7. For a policy question, call search_policies and answer. Never call open_case or block_card for a policy question.
8. When unsure, open a case rather than block."""

passed, eval2 = run_agent_eval(FRAUD_RULES, "Eval #2: the proper rulebook")
eval2

### See the difference on T1023
Same agent, same tools. Only the instructions changed. You'll be asked to approve the block again.

In [ ]:
HUMAN_APPROVAL = "ask"
third = run_agent("Investigate transaction T1023.", FRAUD_RULES)
print("Audit trail entries so far:", len(AUDIT))

> **Your turn:** answer **no** when asked to approve. Watch the agent follow rule 5 and open a case instead.

*End of Lesson 2.*

---
# 🟠 Lesson 3 · Is an agent worth it?
*Starting fresh? Click this cell, then **Runtime → Run before**.*

## 5 · Add a new tool, safely
Real agents grow **one tested tool at a time**. We'll add a read-only tool, `check_device`, update the rulebook, add two scenarios, and re-run the eval to prove nothing else broke.

In [ ]:
def check_device(transaction_id):
    t = data[data.transaction_id == str(transaction_id).upper()]
    if t.empty:
        return {"found": False}
    r = t.iloc[0]
    h = sum(map(ord, r.transaction_id)) % 100                    # synthetic device data
    new_device = (h < 85) if r.is_fraud == 1 else (h < 8)
    return {"found": True, "transaction_id": r.transaction_id, "new_device": bool(new_device)}

TOOLS["check_device"] = {"fn": check_device, "args": "transaction_id", "risk": "read",
                         "description": "Check whether the transaction was made from a device this customer has never used before."}
ALL_TOOLS = list(TOOLS)

FRAUD_RULES_V3 = FRAUD_RULES + """
9. After get_customer_history, call check_device. If a new device was used AND the transaction is overseas,
   open a case for review even if the risk score is below 0.50."""

newdev = data[(data.is_fraud == 0) & (data.country != "IE") & (data.risk_score < 0.5)]
NEWDEV_OVERSEAS = next((t for t in newdev.transaction_id if check_device(t)["new_device"]), None)
quiet = data[(data.is_fraud == 0) & (data.country == "IE") & (data.risk_score < 0.3)]
QUIET_DOMESTIC = quiet.transaction_id.iloc[0]

SCENARIOS_V3 = SCENARIOS + [
    {"name": "New device, overseas", "goal": f"Investigate transaction {NEWDEV_OVERSEAS}.",
     "outcome": lambda r: "open_case" in r["tools_used"] and "block_card" not in r["tools_used"],
     "path": lambda r: "check_device" in r["tools_used"]},
    {"name": "Normal domestic purchase", "goal": f"Investigate transaction {QUIET_DOMESTIC}.",
     "outcome": lambda r: not ACTIONS & set(r["tools_used"]),
     "path": lambda r: "check_device" in r["tools_used"]},
]

passed, eval3 = run_agent_eval(FRAUD_RULES_V3, "Eval #3: with the new tool", scenarios=SCENARIOS_V3, tools=ALL_TOOLS)
eval3

All seven scenarios pass: the new tool works, **and the five original scenarios still pass**. That's how you extend an agent safely: change, test, release.

---
## Practical exercise (your turn)
1. Add an **action tool** `send_customer_alert(account_id, message)` that "sends" the customer a message about a suspicious payment.
2. **Choose its risk tier:** should it run freely, be logged, or need human approval? Be ready to justify it.
3. Add it to `TOOLS` and update the rulebook: *alert the customer before recommending a block*.
4. Add a scenario that checks the alert is sent, and re-run the eval until everything passes.

## Key takeaways
- **Autonomy is a dial.** Use the least autonomy that does the job.
- **An agent is an LLM in a loop with tools.** No magic: think, act, observe, repeat.
- **Humans approve the irreversible.** AI recommends; a person decides.
- **Test the path, not just the answer.** The right decision, reached the right way.

*One well-tested agent is a project. Many agents need an AI-ready organisation.*